In [ ]:
import os
import math
import cv2
import torch
import numpy as np
from numpy import ndarray
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from IPython.display import HTML, display
from ultralytics import YOLO
from typing import Literal, Optional, List

plt.rcParams['animation.embed_limit'] = 250.0
plt.style.use('dark_background')



### Прямой и обратный проход через Multi-Head Attention.

In [ ]:
# НАСТРОЙКИ ПАРАМЕТРОВ
h = 2                                 # Количество голов внимания.
d_model = 256                         # Длина embedding-гов.
batch_size = 5                        # Размер батча.
seq_len = 100                         # Длинна последовательности.
d_q = d_k = d_v = int(d_model / h)
scale = 1 / d_k**0.5

# ИНИЦИАЛИЗАЦИЯ ВЕСОВ
# Первоя голова
wQ_1 = torch.rand(d_model, d_q, requires_grad=True)
wK_1 = torch.rand(d_model, d_k, requires_grad=True)
wV_1 = torch.rand(d_model, d_v, requires_grad=True)

# Вторая голова
wQ_2 = torch.rand(d_model, d_q, requires_grad=True)
wK_2 = torch.rand(d_model, d_k, requires_grad=True)
wV_2 = torch.rand(d_model, d_v, requires_grad=True)

wO = torch.rand(d_model, d_model, requires_grad=True)


In [ ]:
x = torch.rand(batch_size, seq_len, d_model).requires_grad_(True)    # входные данные

"""    ПРЯМОЙ ПРОХОД В РУЧНУЮ    """

# Прямой проход - Первая голова
Q_1 = x @ wQ_1    # (bs, seq_len, d_q)
K_1 = x @ wK_1    # (bs, seq_len, d_k)
V_1 = x @ wV_1    # (bs, seq_len, d_v)

DP1 = Q_1 @ K_1.permute(0, 2, 1)    # (bs, seq_len, seq_len)
SDP1 = DP1 * scale                  # (bs, seq_len, seq_len)

max_SDP1, idx1 = SDP1.max(dim=2, keepdims=True)      # (bs, seq_len, 1)
exp_SDP1 = torch.exp(SDP1 - max_SDP1)             # (bs, seq_len, seq_len)
sum_SDP1 = exp_SDP1.sum(dim=2, keepdims=True)     # (bs, seq_len, 1)
softmax_SDP1 = exp_SDP1 / sum_SDP1                # (bs, seq_len, seq_len)

head_1 = softmax_SDP1 @ V_1    # (bs, seq_len, d_v)

# Прямой проход - Вторая голова
Q_2 = x @ wQ_2    # (bs, seq_len, d_q)
K_2 = x @ wK_2    # (bs, seq_len, d_k)
V_2 = x @ wV_2    # (bs, seq_len, d_v)

DP2 = Q_2 @ K_2.permute(0, 2, 1)    # (bs, seq_len, seq_len)
SDP2 = DP2 * scale                  # (bs, seq_len, seq_len)

max_SDP2, idx2 = SDP2.max(dim=2, keepdims=True)      # (bs, seq_len, 1)
exp_SDP2 = torch.exp(SDP2 - max_SDP2)             # (bs, seq_len, seq_len)
sum_SDP2 = exp_SDP2.sum(dim=2, keepdims=True)     # (bs, seq_len, 1)
softmax_SDP2 = exp_SDP2 / sum_SDP2                # (bs, seq_len, seq_len)

head_2 = softmax_SDP2 @ V_2    # (bs, seq_len, d_v)

# Объединение голов
Atten = torch.concat([head_1, head_2], dim=2)    # (bs, seq_len, d_model)

MHA = Atten @ wO    # (bs, seq_len, d_model)

add_x = x + MHA


In [ ]:
"""    ОБРАТНЫЙ ПРОХОД В РУЧНУЮ    """

d_add_x = torch.ones_like(add_x)    # (bs, seq_len, d_model)

# Обратный проход: add_x = x + MHA
dx = d_add_x.clone()                        # (bs, seq_len, d_model)
dMHA = d_add_x.clone()                      # (bs, seq_len, d_model)

# Обратный проход: MHA = Atten @ wO
dwO = torch.sum(Atten.permute(0, 2, 1) @ dMHA, dim=0)    # (d_model, d_model)
dAtten = dMHA @ wO.T                                     # (bs, seq_len, d_model)

# Обратный проход: Atten = torch.concat([head_1, head_2], dim=2)
dhead_1 = dAtten[:, :, :d_model//h]    # (bs, seq_len, d_v)
dhead_2 = dAtten[:, :, d_model//h:]    # (bs, seq_len, d_v)

""" Обратный проход: Вторая голова """
# Обратный проход: head_2 = softmax_SDP2 @ V_2
dsoftmax_SDP2 = dhead_2 @ V_2.permute(0, 2, 1)    # (bs, seq_len, seq_len)
dV_2 = softmax_SDP2.permute(0, 2, 1) @ dhead_2    # (bs, seq_len, d_v)

# Обратный проход: softmax_SDP2 = exp_SDP2 / sum_SDP2
dexp_SDP2 = dsoftmax_SDP2 / sum_SDP2                     # (bs, seq_len, seq_len)
dsum_SDP2 = torch.sum(                                   # (bs, seq_len, 1)
    dsoftmax_SDP2 * exp_SDP2 / torch.square(sum_SDP2),
    dim=2,
    keepdims=True
)

# Обратный проход: sum_SDP2 = exp_SDP2.sum(dim=2, keepdims=True)
dexp_SDP2 += dsum_SDP2       # (bs, seq_len, seq_len)

# Обратный проход: exp_SDP2 = torch.exp(SDP2 - max_SDP2)
dSDP2 = dexp_SDP2 * torch.exp(SDP2 - max_SDP2)       # (bs, seq_len, seq_len)
dmax_SDP2 = torch.sum(                               # (bs, seq_len, 1)
    -1.0 * dexp_SDP2 * torch.exp(SDP2 - max_SDP2),
    dim=2,
    keepdims=True
)

# Обратный проход: max_SDP2, idx2 = SDP2.max(dim=2, keepdims=True)
# Добавляем dmax_SDP2 ТОЛЬКО в позиции, указанные в idx2
dSDP2.scatter_add_(dim=2, index=idx2, src=dmax_SDP2)    # (bs, seq_len, seq_len)

# Обратный проход: SDP2 = DP2 * scale
dDP2 = dSDP2 * scale    # (bs, seq_len, seq_len)

# Обратный проход: DP2 = Q_2 @ K_2.permute(0, 2, 1)
dQ_2 = dDP2 @ K_2                      # (bs, seq_len, d_q)
dK_2 = dDP2.permute(0, 2, 1) @ Q_2     # (bs, seq_len, d_k)

# Обратный проход: Q_2 = x @ wQ_2,  K_2 = x @ wK_2, V_2 = x @ wV_2
dwQ_2 = torch.sum(x.permute(0, 2, 1) @ dQ_2, dim=0)    # (d_model, d_q)
dwK_2 = torch.sum(x.permute(0, 2, 1) @ dK_2, dim=0)    # (d_model, d_k)
dwV_2 = torch.sum(x.permute(0, 2, 1) @ dV_2, dim=0)    # (d_model, d_v)

dx += dQ_2 @ wQ_2.T + dK_2 @ wK_2.T + dV_2 @ wV_2.T    # (bs, seq_len, d_model)

""" Обратный проход: Первая голова """
# Обратный проход: head_1 = softmax_SDP1 @ V_1
dsoftmax_SDP1 = dhead_1 @ V_1.permute(0, 2, 1)    # (bs, seq_len, seq_len)
dV_1 = softmax_SDP1.permute(0, 2, 1) @ dhead_1    # (bs, seq_len, d_v)

# Обратный проход: softmax_SDP1 = exp_SDP1 / sum_SDP1
dexp_SDP1 = dsoftmax_SDP1 / sum_SDP1                     # (bs, seq_len, seq_len)
dsum_SDP1 = torch.sum(                                   # (bs, seq_len, 1)
    dsoftmax_SDP1 * exp_SDP1 / torch.square(sum_SDP1),
    dim=2,
    keepdims=True
)

# Обратный проход: sum_SDP1 = exp_SDP1.sum(dim=2, keepdims=True)
dexp_SDP1 += dsum_SDP1       # (bs, seq_len, seq_len)

# Обратный проход: exp_SDP1 = torch.exp(SDP1 - max_SDP1)
dSDP1 = dexp_SDP1 * torch.exp(SDP1 - max_SDP1)       # (bs, seq_len, seq_len)
dmax_SDP1 = torch.sum(                               # (bs, seq_len, 1)
    -1.0 * dexp_SDP1 * torch.exp(SDP1 - max_SDP1),
    dim=2,
    keepdims=True
)

# Обратный проход: max_SDP1, idx1 = SDP1.max(dim=2, keepdims=True)
# Добавляем dmax_SDP1 ТОЛЬКО в позиции, указанные в idx1
dSDP1.scatter_add_(dim=2, index=idx1, src=dmax_SDP1)    # (bs, seq_len, seq_len)

# Обратный проход: SDP1 = DP1 * scale
dDP1 = dSDP1 * scale    # (bs, seq_len, seq_len)

# Обратный проход: DP1 = Q_1 @ K_1.permute(0, 2, 1)
dQ_1 = dDP1 @ K_1                      # (bs, seq_len, d_q)
dK_1 = dDP1.permute(0, 2, 1) @ Q_1     # (bs, seq_len, d_k)

# Обратный проход: Q_1 = x @ wQ_1,  K_1 = x @ wK_1, V_1 = x @ wV_1
dwQ_1 = torch.sum(x.permute(0, 2, 1) @ dQ_1, dim=0)    # (d_model, d_q)
dwK_1 = torch.sum(x.permute(0, 2, 1) @ dK_1, dim=0)    # (d_model, d_k)
dwV_1 = torch.sum(x.permute(0, 2, 1) @ dV_1, dim=0)    # (d_model, d_v)

dx += dQ_1 @ wQ_1.T + dK_1 @ wK_1.T + dV_1 @ wV_1.T    # (bs, seq_len, d_model)


In [ ]:
"""    ПРЯМОЙ И ОБРАТНЫЙ ПРОХОД С ПОМОЩЬЮ АВТОГРАД    """

torch_x = x.clone().detach().requires_grad_(True)    # входные данные

# Первоя голова
torch_wQ_1 = wQ_1.clone().detach().requires_grad_(True)
torch_wK_1 = wK_1.clone().detach().requires_grad_(True)
torch_wV_1 = wV_1.clone().detach().requires_grad_(True)

# Вторая голова
torch_wQ_2 = wQ_2.clone().detach().requires_grad_(True)
torch_wK_2 = wK_2.clone().detach().requires_grad_(True)
torch_wV_2 = wV_2.clone().detach().requires_grad_(True)


torch_wO = wO.clone().detach().requires_grad_(True)


# Прямой проход - Первая голова
torch_Q_1 = torch_x @ torch_wQ_1    # (bs, seq_len, d_q)
torch_K_1 = torch_x @ torch_wK_1    # (bs, seq_len, d_k)
torch_V_1 = torch_x @ torch_wV_1    # (bs, seq_len, d_v)

torch_DP1 = torch_Q_1 @ torch_K_1.permute(0, 2, 1)    # (bs, seq_len, seq_len)
torch_SDP1 = torch_DP1 * scale                        # (bs, seq_len, seq_len)

torch_max_SDP1, torch_idx1 = torch_SDP1.max(dim=2, keepdims=True)      # (bs, seq_len, 1)
torch_exp_SDP1 = torch.exp(torch_SDP1 - torch_max_SDP1)             # (bs, seq_len, seq_len)
torch_sum_SDP1 = torch_exp_SDP1.sum(dim=2, keepdims=True)     # (bs, seq_len, 1)
torch_softmax_SDP1 = torch_exp_SDP1 / torch_sum_SDP1                # (bs, seq_len, seq_len)

torch_head_1 = torch_softmax_SDP1 @ torch_V_1    # (bs, seq_len, d_v)

# Прямой проход - Вторая голова
torch_Q_2 = torch_x @ torch_wQ_2    # (bs, seq_len, d_q)
torch_K_2 = torch_x @ torch_wK_2    # (bs, seq_len, d_k)
torch_V_2 = torch_x @ torch_wV_2    # (bs, seq_len, d_v)

torch_DP2 = torch_Q_2 @ torch_K_2.permute(0, 2, 1)    # (bs, seq_len, seq_len)
torch_SDP2 = torch_DP2 * scale                        # (bs, seq_len, seq_len)

torch_max_SDP2, torch_idx2 = torch_SDP2.max(dim=2, keepdims=True)      # (bs, seq_len, 1)
torch_exp_SDP2 = torch.exp(torch_SDP2 - torch_max_SDP2)             # (bs, seq_len, seq_len)
torch_sum_SDP2 = torch_exp_SDP2.sum(dim=2, keepdims=True)     # (bs, seq_len, 1)
torch_softmax_SDP2 = torch_exp_SDP2 / torch_sum_SDP2                # (bs, seq_len, seq_len)

torch_head_2 = torch_softmax_SDP2 @ torch_V_2    # (bs, seq_len, d_v)

torch_Atten = torch.concat([torch_head_1, torch_head_2], dim=2)    # (bs, seq_len, d_model)

torch_MHA = torch_Atten @ torch_wO    # (bs, seq_len, d_model)

torch_add_x = torch_x + torch_MHA

torch_add_x.backward(torch.ones_like(torch_add_x))


In [ ]:
print("\nПроверка прямого прохода\n")
print("torch_add_x == add_x ------->", torch.allclose(torch_add_x, add_x))

print("\nПроверка градиентов для обучаемых параметров\n")

print("torch_wQ_1.grad == dwQ_1 --->", torch.allclose(torch_wQ_1.grad, dwQ_1))
print("torch_wK_1.grad == dwK_1 --->", torch.allclose(torch_wK_1.grad, dwK_1))
print("torch_wV_1.grad == dwV_1 --->", torch.allclose(torch_wV_1.grad, dwV_1))
print("torch_wQ_2.grad == dwQ_2 --->", torch.allclose(torch_wQ_2.grad, dwQ_2))
print("torch_wK_2.grad == dwK_2 --->", torch.allclose(torch_wK_2.grad, dwK_2))
print("torch_wV_2.grad == dwV_2 --->", torch.allclose(torch_wV_2.grad, dwV_2))
print("torch_wO.grad == dwO ------->", torch.allclose(torch_wO.grad, dwO))

print("\ntorch_x.grad == dx --------->", torch.allclose(torch_x.grad, dx))


### Визуализация механизма внимания

In [ ]:
class YOLOAttentionVisualizer:
    def __init__(
        self,
        model_type: Literal['yolo11n', 'yolo11s', 'yolo11m', 'yolo11l', 'yolo11x'] = 'yolo11n',
        image_path: str = '',
        save_video: bool = True,
        save_frames: bool = True,
        show_in_notebook: bool = True,
        target_size: int = 640
    ):
        self.model_type = model_type
        self.model_weights = f"{model_type}.pt"
        self.image_path = image_path
        self.save_video = save_video
        self.save_frames = save_frames
        self.show_in_notebook = show_in_notebook
        self.target_size = target_size

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"🚀 Визуализация механизма внимания [{self.model_type}] на устройстве [{self.device.upper()}]...")
        self.model = self._load_model()
        self.original_img, self.input_tensor = self._preprocess_image()
        self.a_head, self.spatial_shape = self._get_attention_head()

    def _load_model(self):
        """Загрузка модели YOLO."""
        if not os.path.exists(self.model_weights):
            print(f"⬇️  Скачивание весов {self.model_weights}...")
            YOLO(self.model_weights)

        ckpt = torch.load(self.model_weights, map_location=self.device, weights_only=False)
        model = ckpt['model'].float().fuse().eval()
        print("✅  Модель успешно загружена.")
        return model.model

    def _norm(self, x: ndarray):
        min_val = x.min(axis=-1, keepdims=True)
        max_val = x.max(axis=-1, keepdims=True)
        return (x - min_val) / (max_val - min_val + 1e-8)

    def _normalize_image(self, img_array: ndarray):
        img_array = img_array.astype(np.float32) / 255
        mean_val = np.array([0.485, 0.456, 0.406])
        std_val = np.array([[0.229, 0.224, 0.225]])
        return (img_array - mean_val) / (std_val + 1e-8)

    def _preprocess_image(self):
        """Загрузка, изменение размера и нормализация изображения."""
        print(f"    Загрузка изображения: {self.image_path}...")
        im = cv2.imread(self.image_path)
        if im is None:
            raise ValueError(f"❌ Не удалось открыть изображение: {self.image_path}")

        h, w = im.shape[:2]
        print(f"    Оригинальный размер: {im.shape}")

        r = self.target_size / max(h, w)
        new_w = int(round(w * r))
        new_h = int(round(h * r))

        im_resized = cv2.resize(im, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        im_rgb = cv2.cvtColor(im_resized, cv2.COLOR_BGR2RGB)
        img_norm = self._normalize_image(im_rgb)

        img_t = img_norm.transpose(2, 0, 1)
        input_tensor = torch.tensor(img_t, dtype=torch.float32)[None].to(self.device)

        original_img = im_rgb.astype(np.float32) / 255.0
        print(f"    Размер после ресайза: {original_img.shape}")
        print("    Предобработка изображения завершена.")
        return original_img, input_tensor

    def _get_attention_head(self):
        """Получение attention_maps из PSA_layer (index 10)."""
        x = self.input_tensor
        for i in range(10):
            x = self.model[i](x)

        psa = self.model[10]
        cv1_out = psa.cv1(x)

        split_idx = x.shape[1] // 2
        _, tensor_right = cv1_out.split((split_idx, split_idx), dim=1)

        all_attns = []

        for m in psa.m:
            attn_module = m.attn
            bs, C, H, W = tensor_right.shape
            num_heads = C // 64 if C >= 64 else 1
            key_dim = int((C // num_heads) * 0.5)
            val_dim = C // num_heads

            qkv = attn_module.qkv(tensor_right)
            q, k, v = qkv.view(bs, num_heads, key_dim * 2 + val_dim, H * W).split(
                [key_dim, key_dim, val_dim], dim=2
            )

            attn = (q.transpose(-2, -1) @ k) * (key_dim ** -0.5)
            attn = attn.softmax(dim=-1)

            all_attns.append(attn.squeeze(0))

        attentions = torch.cat(all_attns, dim=0)
        print(f"    Извлечено голов внимания: {attentions.shape[0]}. Сетка (HxW): {H}x{W}")
        return attentions, (H, W)

    def _render_head_animation(
        self,
        attentions: torch.Tensor,
        spatial_shape: tuple,
        head_idx: int,
        color_att: str,
        fps: int,
        dpi: int,
        video_name: str,
    ) -> Optional[HTML]:
        """Рендерер анимации для конкретной головы внимания."""

        H, W = spatial_shape
        total_frames = H * W
        num_head = attentions.shape[0]

        print(f"\n🎨 [{head_idx + 1}/{num_head}] Рендеринг головы #{head_idx + 1} (внутренний индекс {head_idx})")

        att_maps = attentions[head_idx].cpu().detach().numpy()    # size(total_frames, H*W)

        H_in, W_in = self.input_tensor.shape[2:]
        h_step, w_step = H_in / H, W_in / W

        fig = plt.figure(figsize=(W_in / dpi, H_in / dpi), dpi=dpi)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.set_axis_off()

        frames_dir = ""
        if self.save_frames:
            frames_dir = f"frames_{self.model_type}_head_{head_idx+1}"
            os.makedirs(frames_dir, exist_ok=True)
            print(f"   📂 Кадры -> {frames_dir}/")

        # Нормализация
        norm_att_maps = self._norm(att_maps)

        ax.imshow(self.original_img)
        rects = [Rectangle((jj * w_step, ii * h_step), w_step, h_step) for ii in range(H) for jj in range(W)]
        pc = PatchCollection(rects, facecolors=color_att, edgecolors='none')
        ax.add_collection(pc)
        red_rect = Rectangle((0, 0), w_step, h_step, edgecolor='red', facecolor='red', lw=1.5)
        ax.add_patch(red_rect)

        print(f"   🎬 Генерация {total_frames} кадров...")

        def animate(idx):
            pc.set_alpha(norm_att_maps[idx])
            red_rect.set_xy(((idx % W) * w_step, (idx // W) * h_step))

            if self.save_frames:
                fig.savefig(f'{frames_dir}/frame_{idx:04d}.png', bbox_inches='tight', pad_inches=0)

            # Прогресс каждые 10%
            if total_frames >= 10 and idx % max(1, total_frames // 10) == 0:
                print(f"      [Прогресс] {idx}/{total_frames} ({100 * idx // total_frames}%)")
            elif idx == total_frames - 1:
                print(f"      [Прогресс] {total_frames}/{total_frames} (100%)")

        ani = animation.FuncAnimation(fig, animate, frames=total_frames, interval=1000 / fps)

        if self.save_video:
            print(f"   🎞️ Склейка видео -> {video_name}...")
            ani.save(video_name, writer='ffmpeg', fps=fps, dpi=dpi)
            print(f"   ✅ Видео сохранено: {video_name}")

        result = None
        if self.show_in_notebook:
            print("   💻 Сборка HTML-плеера для Jupyter...")
            result = HTML(ani.to_jshtml())

        plt.close(fig)
        return result

    def visualize_head(
        self,
        head: int = 1,
        color_att: str = '#FF8017',
        fps: int = 10,
        dpi: int = 150,
        video_name: str = 'attention_viz.mp4',
    ) -> Optional[HTML]:
        """
        Создаёт визуализацию для одной указанной головы внимания.

        Args:
            head:       номер головы (нумерация с 1)
            color_att:  цвет тепловых пятен
            fps:        частота кадров
            dpi:        разрешение
            video_name: имя выходного видеофайла
        """
        print(f"\n🎯 Запуск визуализации одной головы внимания (head={head})...")

        num_head = self.a_head.shape[0]

        if head < 1:
            raise ValueError(f"❌ Нумерация голов начинается с 1. Вы указали head={head}")
        head_idx = head - 1

        if head_idx >= num_head:
            raise ValueError(
                f"❌ В модели {self.model_type} всего {num_head} голов внимания. "
                f"Вы указали head={head}."
            )

        print(f"📊 Всего голов в {self.model_type}: {num_head}")

        return self._render_head_animation(
            attentions=self.a_head,
            spatial_shape=self.spatial_shape,
            head_idx=head_idx,
            color_att=color_att,
            fps=fps,
            dpi=dpi,
            video_name=video_name,
        )

    def visualize_all_heads(
        self,
        color_att: str = '#FF8017',
        fps: int = 10,
        dpi: int = 150,
        output_dir: str = 'attention_videos',
        video_prefix: str = 'attention_head',
    ) -> List[str]:
        """
        Создаёт визуализацию для всех голов внимания модели.

        Args:
            color_att:    цвет тепловых пятен
            fps:          частота кадров
            dpi:          разрешение
            output_dir:   папка, куда созранять видео
            video_prefix: префикс имён файлов

        Returns:
            Список путей к созданным видеофайлам.
        """
        num_head = self.a_head.shape[0]

        print(f"\n{'=' * 60}")
        print(f"🎯 ВИЗУАЛИЗАЦИЯ ВСЕХ ГОЛОВ ВНИМАНИЯ")
        print(f"   Модель:       {self.model_type}")
        print(f"   Голов всего:  {num_head}")
        print(f"   Папка вывода: {output_dir}/")
        print(f"{'=' * 60}")

        if self.save_video:
            os.makedirs(output_dir, exist_ok=True)

        created_files: List[str] = []
        width = len(str(num_head))  # для паддинга в именах файлов (01, 02, ...)

        for head_idx in range(num_head):
            print(f"\n{'─' * 60}")
            print(f"▶️  ГОЛОВА {head_idx + 1} из {num_head}")
            print(f"{'─' * 60}")

            video_name = os.path.join(
                output_dir,
                f"{video_prefix}_{str(head_idx + 1).zfill(width)}.mp4",
            )

            self._render_head_animation(
                attentions=self.a_head,
                spatial_shape=self.spatial_shape,
                head_idx=head_idx,
                color_att=color_att,
                fps=fps,
                dpi=dpi,
                video_name=video_name,
            )

            if self.save_video:
                created_files.append(video_name)

        print(f"\n{'=' * 60}")
        print(f"🏁 ГОТОВО! Все {num_head} голов обработаны.")
        if created_files:
            print("📁 Созданные файлы:")
            for f in created_files:
                print(f"   • {f}")
        print(f"{'=' * 60}")

        return created_files

    def visualize_all_heads_combined(
        self,
        color_att: str = '#FF8017',
        fps: int = 10,
        dpi: int = 120,
        ncols: Optional[int] = None,
        video_name: str = 'all_heads_combined.mp4',
        frames_dir_name: Optional[str] = None,
    ) -> Optional[HTML]:
        """
        Создаёт ОДНУ анимацию, в которой все головы внимания отображаются
        одновременно в виде сетки subplot'ов с минимальными отступами.
        """
        num_head = self.a_head.shape[0]
        H, W = self.spatial_shape
        H_in, W_in = self.input_tensor.shape[2:]
        h_step, w_step = H_in / H, W_in / W
        total_frames = H * W

        # Рассчитываем оптимальную сетку
        if ncols is None:
            ncols = math.ceil(math.sqrt(num_head))
        nrows = math.ceil(num_head / ncols)

        print(f"\n{'=' * 60}")
        print(f"🎯 КОМБИНИРОВАННАЯ ВИЗУАЛИЗАЦИЯ ВСЕХ ГОЛОВ")
        print(f"   Модель:           {self.model_type}")
        print(f"   Голов всего:      {num_head}")
        print(f"   Сетка (rows x cols): {nrows} x {ncols}")
        print(f"   Кадров на голову: {total_frames}")
        print(f"{'=' * 60}")

        # Создаём figure с минимальными отступами
        subplot_w = W_in / dpi
        subplot_h = H_in / dpi

        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(subplot_w * ncols, subplot_h * nrows),
            dpi=dpi,
            gridspec_kw={
                'wspace': 0.005,  # Минимальное пространство между колонками (~5px)
                'hspace': 0.005,  # Минимальное пространство между строками (~5px)
            }
        )

        # Приводим axes к плоскому списку
        axes_flat = np.array(axes).flatten() if num_head > 1 else np.array([axes])

        # Убираем пустые subplot'ы
        for idx in range(num_head, nrows * ncols):
            axes_flat[idx].axis('off')
            axes_flat[idx].set_visible(False)

        a_head = self.a_head.cpu().detach().numpy()    # size(num_head, total_frames, H*W)

        head_data = self._norm(a_head)

        # Настраиваем каждый subplot с минимальными отступами
        patch_collections = []
        red_rects = []

        for head_idx in range(num_head):
            ax = axes_flat[head_idx]
            ax.set_axis_off()
            ax.imshow(self.original_img)
            ax.margins(0)  # Убираем внутренние отступы

            # Тепловая карта
            rects = [
                Rectangle((jj * w_step, ii * h_step), w_step, h_step)
                for ii in range(H) for jj in range(W)
            ]
            pc = PatchCollection(rects, facecolors=color_att, edgecolors='none')
            ax.add_collection(pc)
            patch_collections.append(pc)

            # Красный индикатор
            red_rect = Rectangle(
                (0, 0), w_step, h_step,
                edgecolor='red', facecolor='red', lw=1.5
            )
            ax.add_patch(red_rect)
            red_rects.append(red_rect)

        # Дополнительно убираем все внешние отступы figure
        plt.subplots_adjust(
            left=0, right=1, top=1, bottom=0,
            wspace=0.005, hspace=0.005
        )

        # Папка для кадров
        frames_dir = ""
        if self.save_frames:
            frames_dir = frames_dir_name or f"frames_{self.model_type}_combined"
            os.makedirs(frames_dir, exist_ok=True)
            print(f"📂 Кадры -> {frames_dir}/")

        # Функция анимации
        print(f"🎬 Генерация {total_frames} кадров (все головы синхронно)...")

        def animate(idx):
            for head_idx in range(num_head):
                patch_collections[head_idx].set_alpha(head_data[head_idx, idx])
                red_rects[head_idx].set_xy(
                    ((idx % W) * w_step, (idx // W) * h_step)
                )

            if self.save_frames:
                fig.savefig(
                    f'{frames_dir}/frame_{idx:04d}.png',
                    bbox_inches='tight',
                    pad_inches=0.01  # Минимальный padding при сохранении
                )

            # Прогресс
            if total_frames >= 10 and idx % max(1, total_frames // 10) == 0:
                pct = 100 * idx // total_frames
                print(f"   [Прогресс] {idx}/{total_frames} ({pct}%)")
            elif idx == total_frames - 1:
                print(f"   [Прогресс] {total_frames}/{total_frames} (100%)")

        ani = animation.FuncAnimation(
            fig, animate, frames=total_frames, interval=1000 / fps
        )

        # 9) Сохранение видео
        if self.save_video:
            print(f"\n🎞️  Склейка видео -> {video_name}...")
            ani.save(video_name, writer='ffmpeg', fps=fps, dpi=dpi)
            print(f"✅ Видео сохранено: {video_name}")

        # 10) Возврат для Jupyter
        result = None
        if self.show_in_notebook:
            print("💻 Сборка HTML-плеера для Jupyter...")
            result = HTML(ani.to_jshtml())

        plt.close(fig)
        print(f"\n🏁 Комбинированная визуализация завершена.")
        return result



In [ ]:
visualizer = YOLOAttentionVisualizer(
    model_type='yolo11n',     # 'yolo11n', 'yolo11s', 'yolo11m', 'yolo11l', 'yolo11x'
    image_path='img_640.jpg',
    save_video=True,
    save_frames=False,
    show_in_notebook=False,
)


In [ ]:
# Визуализация одной головы
visualizer.visualize_head(
    head=1,
    color_att='#2F6DFC',
    video_name='yolo11n_head_01_car.mp4'
)


In [ ]:
# Визуализация всех голов сразу
visualizer.visualize_all_heads(
    output_dir='yolo11n_all_heads_car',
    color_att='#7030A0'
)


In [ ]:
# 'yolo11n':  2 head
# 'yolo11s':  4 head
# 'yolo11m':  4 head
# 'yolo11l':  8 head
# 'yolo11x': 12 head

# Все головы в одной сетке
visualizer.visualize_all_heads_combined(
    color_att='#7030A0',
    ncols=2,    # строки посчитаются автоматически
    video_name='yolo11n_all_heads_car.mp4',
)